# 04 - Learning-to-Rank (LightGBM LambdaMART)
This notebook builds and trains the second-stage ranking model:
- Re-ranking candidate items retrieved from ALS Matrix Factorization
- Generating negative samples & building session query groups
- Training LightGBM with `lambdarank` objective
- Evaluating ranking with NDCG@10 and MAP@10
- Feature importance and SHAP interpretability

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import joblib
import yaml
import shap
import matplotlib.pyplot as plt
from pathlib import Path

# Load project configurations
with open("../config.yaml", "r") as f:
    config = yaml.safe_load(f)

models_dir = Path("../models")
results_dir = Path("../results")
models_dir.mkdir(exist_ok=True)
results_dir.mkdir(exist_ok=True)
print("Environment and configs ready.")


### 1. Construct Ranking Query Groups & Training Data

In [ ]:
# Simulating/loading candidate pairs with interaction labels and ranking features
# Features: als_score, user_activity, item_popularity, interaction_recency
np.random.seed(42)
n_queries = 2000
candidates_per_query = 50

user_ids = np.repeat(np.arange(n_queries), candidates_per_query)
item_ids = np.random.randint(1000, 5000, size=len(user_ids))

# Feature engineering columns
als_scores = np.random.uniform(0.1, 1.0, size=len(user_ids))
user_activity = np.repeat(np.random.poisson(5, size=n_queries), candidates_per_query)
item_popularity = np.random.poisson(20, size=len(user_ids))
interaction_recency = np.random.exponential(10, size=len(user_ids))

# Synthetic relevance labels: 0 = impression, 1 = view, 2 = cart, 3 = transaction
relevance_logits = 2.5 * als_scores + 0.05 * item_popularity - 0.02 * interaction_recency
probs = 1 / (1 + np.exp(-relevance_logits + 3.0))
labels = np.random.binomial(3, p=np.clip(probs, 0.0, 1.0))

rank_df = pd.DataFrame({
    'user_id': user_ids,
    'item_id': item_ids,
    'als_score': als_scores,
    'user_activity': user_activity,
    'item_popularity': item_popularity,
    'interaction_recency': interaction_recency,
    'target': labels
})

# Split train and validation by query users
train_split_user = int(n_queries * 0.8)
train_df = rank_df[rank_df['user_id'] < train_split_user]
val_df = rank_df[rank_df['user_id'] >= train_split_user]

feature_cols = ['als_score', 'user_activity', 'item_popularity', 'interaction_recency']

X_train, y_train = train_df[feature_cols], train_df['target']
X_val, y_val = val_df[feature_cols], val_df['target']

# Calculate group sizes (number of candidates per query user)
train_groups = train_df.groupby('user_id').size().values
val_groups = val_df.groupby('user_id').size().values

print(f"Train samples: {len(X_train):,} across {len(train_groups):,} queries")
print(f"Validation samples: {len(X_val):,} across {len(val_groups):,} queries")


### 2. Train LightGBM LambdaMART Ranker

In [ ]:
ranker = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    eval_at=[5, 10],
    n_estimators=150,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42
)

ranker.fit(
    X_train,
    y_train,
    group=train_groups,
    eval_set=[(X_val, y_val)],
    eval_group=[val_groups],
    callbacks=[lgb.early_stopping(stopping_rounds=15), lgb.log_evaluation(period=25)]
)

# Persist trained model
model_path = models_dir / "ranker_model.txt"
ranker.booster_.save_model(str(model_path))
print(f"Trained model persisted to: {model_path}")


### 3. Feature Importance & SHAP Interpretability

In [ ]:
# Plot Split and Gain Feature Importance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
lgb.plot_importance(ranker, importance_type="split", ax=axes[0], title="Feature Importance (Split)")
lgb.plot_importance(ranker, importance_type="gain", ax=axes[1], title="Feature Importance (Gain)")
plt.tight_layout()
plt.savefig(results_dir / "lgbm_feature_importance.png")
plt.show()

# SHAP Tree Explainer
explainer = shap.TreeExplainer(ranker.booster_)
shap_values = explainer.shap_values(X_val.sample(500, random_state=42))

plt.figure()
shap.summary_plot(shap_values, X_val.sample(500, random_state=42), plot_type="bar", show=False)
plt.title("Mean Absolute SHAP Values across Ranking Features")
plt.tight_layout()
plt.savefig(results_dir / "lgbm_shap_summary.png")
plt.show()
